# Hyperparameter Tuning

## Goals

The overarching goal is to learn to use callbacks for some typical tasks. These include:
- Reporting about training progress.
- Stoping once training no longer reduces loss.
- Tuning hyperparameters.
- Implementing adaptive learning rate decay.
- Finding an optimal batch-size for training.
- Putting some of this into ```my_keras_utils.py``` so that they can be easily called and reused.

## What's Here?

I continue working with MNIST data, which I began working with in [my first Keras models](first_model.ipynb). 

My **concrete objective** is to tune a model that does well on Kaggle: 97th percentile? That's tough, but I think I can make it work.

In [1]:
import numpy as np
from datetime import datetime, time, timedelta

import pandas as pd
import tensorflow as tf
import kerastuner as kt
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt


import my_keras_utils as my_utils

In [2]:
tf.__version__
tf.config.experimental.list_physical_devices('GPU')

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [3]:
## Load our data.
## Since the load process is a little slow, the try-except allows us to re-run all 
## cells without having to wait. 
try:
    ## Raises NameError and loads data if X_train is not defined.
    X_train.shape
except NameError: 
    ((X_train, y_train), (X_dev, y_dev), (X_test, y_test)) = my_utils.load_kaggle_mnist()

## Reshape the flattened rows into (-1,28,28,1)
## For augmentation
X_train = X_train.reshape((-1,28,28,1))    
X_dev = X_dev.reshape(-1,28,28,1)
X_test = X_test.reshape(-1,28,28,1)

In [4]:

def overlay_histories(histories, metric):
    fig, ax = plt.subplots()
    n = 0
    for h in histories:
        x = range(0,len(h.history[metric]))
        y = np.array(h.history[metric])
        label = 'history_' + str(n)
        ax.plot(x,y, label=label)
        n += 1
    ax.legend()

## Augmentation Model

In [27]:
augment_model = keras.models.Sequential(name= 'augment_layer')
augment_model.add(layers.experimental.preprocessing
                    .RandomRotation(factor = 1./20.,
                                    fill_mode = 'constant'))
augment_model.add(layers.experimental.preprocessing
                    .RandomTranslation(height_factor=2./28.,
                                    width_factor= 2./28.,
                                    fill_mode = 'constant'))
augment_model.add(layers.Flatten())

## Tuning

Time to work with Keras Tuner.



In [6]:
def model_builder(hp):
    ## ### I need to learn about the options here. 'Choice' means "here are your choices"
    ## ### 'Int' is a different option that searches an integer range by steps.

    inputs = keras.Input(shape=(28,28,1))
    x = augment_model(inputs)
    x = layers.experimental.preprocessing.Rescaling(1./255)(x)
    x = layers.Dropout(rate = hp.Float('drop_rate_0',
                                min_value = .0,
                                max_value = .45,
                                sampling = 'linear',
                                default = .25))(x)
    for i in range(hp.Int('num_layers', min_value = 2, max_value = 3, default = 2)):
        hp_units = hp.Int('units_l'+str(i+1), 
                            min_value = 80, 
                            max_value = 120, 
                            step = 1,
                            default = 100)
        x = layers.Dense(hp_units, activation = 'relu')(x)
        hp_dropout = hp.Float('drop_rate_'+str(i+1),
                                min_value = .0,
                                max_value = .45,
                                sampling = 'linear',
                                default = .15)
        x = layers.Dropout(rate=hp_dropout)(x)
    outputs = layers.Dense(10, activation='softmax')(x)
    model = keras.Model(inputs=inputs, outputs=outputs)

    model.compile(optimizer = keras.optimizers
                                .Adam(hp.Float('learning_rate',
                                                min_value = .0003,
                                                max_value = .01,
                                                sampling = 'log',
                                                default = .001)),
                    loss = "sparse_categorical_crossentropy",
                    run_eagerly = False,
                    metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")]
    )

    return model

	


In [28]:
## Tuner callbacks
clear_output = my_utils.ClearTrainingOutput()
timed_update = my_utils.TimedProgressUpdate(3)
train_loss_stopping = keras.callbacks.EarlyStopping(monitor='loss', 
                            patience = 10, 
                            restore_best_weights = False
                            )
val_loss_stopping = keras.callbacks.EarlyStopping(monitor='val_loss', 
                            patience = 10, 
                            restore_best_weights = False
                            )

adaptive_lr = keras.callbacks.ReduceLROnPlateau(
                    monitor='loss',
                    patience= 10,
                    cooldown= 2,
                    min_lr=.00005, 
                    factor=0.5,)

tuner_callbacks = [adaptive_lr, train_loss_stopping, val_loss_stopping, clear_output]

In [14]:

hp = kt.HyperParameters()
hp.Float('learning_rate',
            min_value = .0002,
            max_value = .0008,
            sampling = 'log',
            default = .001)
hp.Fixed('num_layers', 3)
hp.Fixed('units_l1', 120)
hp.Fixed('units_l2', 100)

tuner = kt.BayesianOptimization(model_builder,
                objective = 'val_loss',
                hyperparameters= hp,
                max_trials = 75,
                executions_per_trial = 1,
                tune_new_entries = True,
                directory = os.path.normpath('C:\mnist'),
                project_name = 'bz',
                overwrite = False)
                

##assert 'keepgoing' == False

tuner.search(X_train,
                y_train,
                validation_data = (X_dev, y_dev),
                epochs = 200,
                batch_size = 64,
                verbose = 0,
                callbacks = [tuner_callbacks])

INFO:tensorflow:Oracle triggered exit


In [15]:
tuner.results_summary()

In [29]:
hp = kt.HyperParameters()
hp.Float('learning_rate',
            min_value = .0002,
            max_value = .0008,
            sampling = 'log',
            )
hp.Float('drop_rate_0', min_value=.0, max_value=.27)            
hp.Float('drop_rate_1', min_value=.0, max_value=.15)
hp.Float('drop_rate_2', min_value=.0, max_value=.15)
hp.Float('drop_rate_3', min_value=.0, max_value=.15)
#hp.Fixed('drop_rate_0', 0)
#hp.Fixed('num_layers', 2)
hp.Fixed('units_l1', 120)
hp.Fixed('units_l2', 120)

tuner5 = kt.BayesianOptimization(model_builder,
                objective = 'val_loss',
                hyperparameters= hp,
                max_trials = 12,
                num_initial_points=3,
                executions_per_trial = 1,
                tune_new_entries = True,
                directory = os.path.normpath('C:\mnist'),
                project_name = 'bz6',
                overwrite = False)
                

##assert 'keepgoing' == False

tuner5.search(X_train,
                y_train,
                validation_data = (X_dev, y_dev),
                epochs = 400,
                batch_size = 64,
                verbose = 0,
                callbacks = [adaptive_lr, clear_output])

INFO:tensorflow:Oracle triggered exit


In [30]:
#tuner2.search_space_summary()
tuner5.results_summary()

In [ ]:
|-drop_rate_0: 0.12293824152640359
|-drop_rate_1: 0.010395574155992644
|-drop_rate_2: 0.12864297589906631
|-drop_rate_3: 0.03723940790532728
|-learning_rate: 0.0007829421833955832
|-num_layers: 3
|-units_l1: 120
|-units_l2: 120
|-units_l3: 99

In [ ]:

inputs = keras.Input(shape=(28,28,1))
x = augmentation_model(inputs)
x = layers.experimental.preprocessing.Rescaling(1./255)(x)
x = layers.Dropout(rate=.12)(x)
x = layers.Dense(120, activation='relu')(x)
x = layers.Dropout(rate=.01)(x)
x = layers.Dense(120, activation='relu')(x)
x = layers.Dropout(rate=.13)(x)
x = layers.Dense(99, activation='relu')(x)
x = layers.Dropout(rate=.0008)(x)
output = layers.Dense(10, activation='softmax')

mnist_model = keras.Model(name='MNIST_classifier', 
            inputs=inputs, 
            output=output)
optimizer = keras.optimizers.Adam(.00078,
                loss = 'sparse_categorical_crossentropy',
                metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")])
mnist_model.compile(optimizer=optimizer)

mnist_model.model_summary()